## Enabling GPU

In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Unzipping Dataset

In [ ]:
from google.colab import drive

import os
drive.mount('/content/drive')
!unzip -q "/content/drive/MyDrive/datasets.zip" -d "/content/"
print("Unzip complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
replace /content/datasets/face_dataset/test/fake/002TJAKUYX.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: Unzip complete!


## Dataset Validation

In [ ]:

BASE_DIR = "/content/datasets/global_dataset"
train_dir = os.path.join(BASE_DIR, "train")
val_dir = os.path.join(BASE_DIR, "val")
test_dir = os.path.join(BASE_DIR, "test")

for d in [train_dir, val_dir, test_dir]:
    if os.path.exists(d):
        print(f"✅ Found: {d}")
        print(f"   Contains: {os.listdir(d)}")
    else:
        print(f"❌ NOT Found: {d}")

✅ Found: /content/datasets/global_dataset/train
   Contains: ['fake', 'real']
✅ Found: /content/datasets/global_dataset/val
   Contains: ['fake', 'real']
✅ Found: /content/datasets/global_dataset/test
   Contains: ['fake', 'real']


## Defining Variables

In [ ]:
IMG_SIZE=(224,224)
EPOCHS=20
BATCH_SIZE=32

## Data Generators

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

val_gen = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

print("Class indices:", train_gen.class_indices)


Found 34673 images belonging to 2 classes.
Found 13318 images belonging to 2 classes.
Class indices: {'fake': 0, 'real': 1}


## MobileNetV2 Model

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D

base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False  # feature extraction first

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## Model Compile

In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


## Callback Created

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

checkpoint = ModelCheckpoint(
    "/content/drive/MyDrive/mobilenetv2_global_best.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)


## Model Training

In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=[checkpoint, early_stop]
)


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 0s 452ms/step - accuracy: 0.7151 - loss: 0.5548
Epoch 1: val_accuracy improved from -inf to 0.79321, saving model to /content/drive/MyDrive/mobilenetv2_global_best.keras
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 553s 491ms/step - accuracy: 0.7152 - loss: 0.5547 - val_accuracy: 0.7932 - val_loss: 0.4347
Epoch 2/20
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 0s 436ms/step - accuracy: 0.8046 - loss: 0.4234
Epoch 2: val_accuracy improved from 0.79321 to 0.80590, saving model to /content/drive/MyDrive/mobilenetv2_global_best.keras
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 498s 459ms/step - accuracy: 0.8046 - loss: 0.4234 - val_accuracy: 0.8059 - val_loss: 0.4180
Epoch 3/20
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 0s 437ms/step - accuracy: 0.8233 - loss: 0.3887
Epoch 3: val_accuracy improved from 0.80590 to 0.82017, saving model to /content/drive/MyDrive/mobilenetv2_global_best.keras
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 500s 462ms/step - accuracy: 0.8233 - loss: 0.3887 - val_accuracy: 0.8202 - val_lo

## Model Tuning

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=[checkpoint, early_stop]
)


Epoch 1/10
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 0s 449ms/step - accuracy: 0.8458 - loss: 0.3473
Epoch 1: val_accuracy improved from 0.84848 to 0.86439, saving model to /content/drive/MyDrive/mobilenetv2_global_best.keras
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 539s 481ms/step - accuracy: 0.8458 - loss: 0.3472 - val_accuracy: 0.8644 - val_loss: 0.3276
Epoch 2/10
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 0s 440ms/step - accuracy: 0.8751 - loss: 0.2940
Epoch 2: val_accuracy improved from 0.86439 to 0.87123, saving model to /content/drive/MyDrive/mobilenetv2_global_best.keras
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 504s 465ms/step - accuracy: 0.8751 - loss: 0.2940 - val_accuracy: 0.8712 - val_loss: 0.3079
Epoch 3/10
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 0s 442ms/step - accuracy: 0.8894 - loss: 0.2711
Epoch 3: val_accuracy did not improve from 0.87123
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 506s 466ms/step - accuracy: 0.8894 - loss: 0.2711 - val_accuracy: 0.8600 - val_loss: 0.3455
Epoch 4/10
1084/1084 ━━━━━━━━━━━━━━━━━━━━ 0s 438ms/step - ac

## Model Save

In [ ]:
model.save("/content/drive/MyDrive/mobilenetv2_global_final_tuned.keras")
print("✅ Model saved to Google Drive")


✅ Model saved to Google Drive


In [28]:
!pip freeze

absl-py==1.4.0
accelerate==1.12.0
access==1.1.10.post3
affine==2.4.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.3
aiosignal==1.4.0
aiosqlite==0.22.1
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
alembic==1.18.1
altair==5.5.0
annotated-doc==0.0.4
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.12.1
anywidget==0.9.21
apsw==3.51.2.0
apswutils==0.1.2
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.3
arrow==1.4.0
arviz==0.22.0
astropy==7.2.0
astropy-iers-data==0.2026.1.26.0.43.56
astunparse==1.6.3
atpublic==5.1
attrs==25.4.0
audioread==3.1.0
Authlib==1.6.6
autograd==1.8.0
babel==2.17.0
backcall==0.2.0
beartype==0.22.9
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.33.0
bigquery-magics==0.10.3
bleach==6.3.0
blinker==1.9.0
blis==1.3.3
blobfile==3.1.0
blosc2==3.12.2
bokeh==3.7.3
Bottleneck==1.4.2
bqplot==0.12.45
branca==0.8.2
brotli==1.2.0
CacheControl==0.14.4
cachetools==6.2.6
catalogue==2.0.10
certifi==2026.1.4
c

In [29]:
!pip freeze | grep -E "tensorflow|keras|numpy|scikit|matplotlib|Pillow|seaborn"


keras==3.10.0
keras-hub==0.21.1
keras-nlp==0.21.1
matplotlib==3.10.0
matplotlib-inline==0.2.1
matplotlib-venn==1.1.2
numpy==2.0.2
scikit-image==0.25.2
scikit-learn==1.6.1
seaborn==0.13.2
tensorflow==2.19.0
tensorflow-datasets==4.9.9
tensorflow-hub==0.16.1
tensorflow-metadata==1.17.3
tensorflow-probability==0.25.0
tensorflow-text==2.19.0
tensorflow_decision_forests==1.12.0
tf_keras==2.19.0
